# PAIR-R24M - Dataset conversion

Data from Marshalö et al. (2024)¹ ([figshare](https://figshare.com/articles/dataset/pairs_dataset/14754374)), a multi-animal 3D pose dataset about the dyadic interactions in laboratory rats.

---

¹ Marshall, J., Klibaite, U., Gellis, A., Aldarondo, D., Olveczky, B., & Dunn, T. W. (2021). The PAIR-R24M Dataset for Multi-animal 3D Pose Estimation. Proceedings of the Neural Information Processing Systems Track on Datasets and Benchmarks, 1. https://datasets-benchmarks-proceedings.neurips.cc/paper/2021/hash/1ff8a7b5dc7a7d1f0ed65aaa29c04b1e-Abstract-round1.html



In [1]:
import numpy as np
import xarray as xr
from pathlib import Path

import ethograph as eto
from movement.kinematics import compute_velocity, compute_speed
from movement.utils.vector import compute_norm

from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
# Download data from https://figshare.com/articles/dataset/pairs_dataset/14754374, specify path to markerDataset.csv from a session
path = r"C:\Users\aksel\Documents\Code\EthoGraph\data\20210119_Recording_SR1_SR2_social_vidtwo\markerDataset.csv"

CHUNK_SIZE = 3500
VIDEO_DIR = Path("data/20210119_Recording_SR1_SR2_social_vidtwo/videos")
CAMERAS = ["Camera1", "Camera2", "Camera3", "Camera4", "Camera5", "Camera6"]
N_FRAMES = 213500
FPS = 120

In [ ]:

def from_pair24_csv(
    file_path: Path | str,
    fps: Optional[float] = None,
) -> xr.Dataset:
    
    df = pd.read_csv(file_path)
    
    # Extract metadata
    keypoint_names = [
        "HeadF", "HeadB", "HeadL", "SpineF", "SpineM", "SpineL",
        "Offset1", "Offset2", "HipL", "HipR", "ShoulderL", "ShoulderR"
    ]
    individual_names = ["an1", "an2"]
    position_types = ["aligned", "absolute"]
    n_frames = len(df)
    n_keypoints = len(keypoint_names)
    n_individuals = len(individual_names)
    n_space = 3  # x, y, z coordinates
    n_position_types = len(position_types)
    
    # Initialize arrays with additional dimension for position type
    position_array = np.zeros((n_frames, n_position_types, n_space, n_keypoints, n_individuals))
    # Fix: confidence array should also have position_type dimension
    confidence_array = np.ones((n_frames, n_position_types, n_keypoints, n_individuals))

    
    # Fill position data for both aligned and absolute
    for p, pos_type in enumerate(position_types):
        # Map our position types to the CSV column prefixes
        csv_prefix = "alignedPosition" if pos_type == "aligned" else "absolutePosition"
        
        for i, individual in enumerate(individual_names):
            for j, keypoint in enumerate(keypoint_names):
                for k, coord in enumerate(["x", "y", "z"]):
                    col_name = f"{csv_prefix}_{individual}_{keypoint}_{coord}"
                    if col_name in df.columns:
                        position_array[:, p, k, j, i] = df[col_name].values


    time_coords = np.arange(n_frames, dtype=float) / fps

    
    # Create base dataset with position_type as a coordinate
    ds = xr.Dataset(
        data_vars={
            "position": xr.DataArray(
                position_array,
                dims=["time", "position_type", "space", "keypoints", "individuals"],
            ),
            "confidence": xr.DataArray(
                confidence_array,
                dims=["time", "position_type", "keypoints", "individuals"], # confidence across space
            ),
        },
        coords={
            "time": time_coords,
            "position_type": position_types,
            "space": ["x", "y", "z"],
            "keypoints": keypoint_names,
            "individuals": ["mouse 1", "mouse 2"],
        },
        attrs={
            "source_software": "DeepLabCut", # for compatibility with movement napari
            "ds_type": "poses",
        }
    )
    
    ds.attrs["fps"] = fps
    
    # Add behavioral annotations
    behavior_mapping = {
        "behaviorCoarse_an1": 0,
        "behaviorCoarse_an2": 1,
    }
    behavior_coarse = np.full((n_frames, n_individuals), np.nan)

    for col, ind_idx in behavior_mapping.items():
        if col in df.columns:
            behavior_coarse[:, ind_idx] = df[col].values
    
    
    # Add center of mass data if present
    com_data = np.zeros((n_frames, n_space, n_individuals))
    for i, individual in enumerate(individual_names):
        for j, coord in enumerate(["x", "y", "z"]):
            col_name = f"centerOfmass_{individual}_{coord}"
            if col_name in df.columns:
                com_data[:, j, i] = df[col_name].values
    
    ds["center_of_mass"] = xr.DataArray(
        com_data,
        dims=["time", "space", "individuals"]
    )
    
    return ds

def get_video_paths_for_chunk(start_frame: int, cameras: list[str]) -> list[str]:
    """Get video file paths for a chunk based on its starting frame."""
    video_paths = []
    for camera in cameras:
        video_path = Path(camera) / f"{start_frame}.mp4"
        video_paths.append(str(video_path))
    return video_paths


def split_dataset_into_chunks(
    ds_full: xr.Dataset,
    chunk_size: int,
    cameras: list[str],
    fps: int,
) -> tuple[list[xr.Dataset], list[list[str]]]:
    """Split a full dataset into fixed-size chunks with video assignments."""
    n_frames = ds_full.sizes["time"]
    n_chunks = n_frames // chunk_size
    
    datasets = []
    all_video_paths = []
    
    for i in range(n_chunks):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        start_frame = start_idx  # Frame number for video filename
        
        # Slice the dataset
        ds_chunk = ds_full.isel(time=slice(start_idx, end_idx)).copy()
        
        # Reset time coordinate to start from 0 for each chunk
        ds_chunk = ds_chunk.assign_coords(time=np.arange(chunk_size) / fps)
        
        # Set trial number
        ds_chunk.attrs["trial"] = i

        
        # Collect video file paths for session-level storage
        video_paths = get_video_paths_for_chunk(start_frame, cameras)
        all_video_paths.append(video_paths)
        

        ds_chunk["pairwise_distance"] = compute_norm(ds_chunk.center_of_mass.sel(individuals="mouse 1") - 
                                                     ds_chunk.center_of_mass.sel(individuals="mouse 2"))
        ds_chunk["pairwise_distance"].attrs["type"] = "features"
        
        ds_chunk["nose_nose_distance"] = compute_norm(
            ds_chunk.position.sel(keypoints='HeadF', individuals="mouse 1", position_type='absolute') -
            ds_chunk.position.sel(keypoints='HeadF', individuals="mouse 2", position_type='absolute')
        )
        ds_chunk["velocity"] = compute_velocity(ds_chunk.position.sel(position_type='aligned'))        
        ds_chunk["speed"] = compute_speed(ds_chunk.position.sel(position_type='aligned'))


        
        # Since we don't all ds.data_vars to be feature variables, we manually specify them here.
        for feature in ["nose_nose_distance", "velocity", "speed"]:
            ds_chunk[feature].attrs["type"] = "features"


        datasets.append(ds_chunk)
        
    remaining = n_frames % chunk_size
    if remaining > 0:
        print(f"Discarded {remaining} frames at the end (not a full chunk)")
    
    print(f"Created {len(datasets)} chunks of {chunk_size} frames each")
    return datasets, all_video_paths

In [12]:
# Conver the .csv into one large xarray dataset
ds_full = from_pair24_csv(path,fps=FPS)
ds_full

<xarray.Dataset> Size: 172MB
Dimensions:         (time: 108000, position_type: 2, space: 3, keypoints: 12,
                     individuals: 2)
Coordinates:
  * time            (time) float64 864kB 0.0 0.008333 0.01667 ... 900.0 900.0
  * position_type   (position_type) <U8 64B 'aligned' 'absolute'
  * space           (space) <U1 12B 'x' 'y' 'z'
  * keypoints       (keypoints) <U9 432B 'HeadF' 'HeadB' ... 'ShoulderR'
  * individuals     (individuals) <U7 56B 'mouse 1' 'mouse 2'
Data variables:
    position        (time, position_type, space, keypoints, individuals) float64 124MB ...
    confidence      (time, position_type, keypoints, individuals) float64 41MB ...
    center_of_mass  (time, space, individuals) float64 5MB -183.0 ... 83.7
Attributes:
    source_software:  DeepLabCut
    ds_type:          poses
    fps:              120

In [ ]:
# Split the dataset, so we have feature data for each video separately
datasets, all_video_paths = split_dataset_into_chunks(
    ds_full,
    chunk_size=CHUNK_SIZE,
    cameras=CAMERAS,
    fps=FPS,
)

In [ ]:
# Create TrialTree from datasets
dt = eto.from_datasets(datasets)
dt.set_media_files(video=all_video_paths, cameras=CAMERAS)
print(f"Created TrialTree with {len(dt.trials)} trials")
print(f"Trial numbers: {dt.trials}")